# Language Models and Author Identification

In [17]:
# Dependencies
import os
import re
import math
from pathlib import Path
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from collections import Counter
import random
from lm_utils import (
    find_base_dir,
    clean_text,
    split_train_holdout,
    get_sentences,
    encode_sentences,
    BPETokenizer,
    NGramModel,
    save_model,
    load_model,
    predict_author,
)

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
MODEL_DIR = BASE_DIR / "models"
DATA_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)
print(f"BASE_DIR resolved to: {BASE_DIR}")

BASE_DIR resolved to: /Users/amberteetsel/MSDS/NLP/nlp-author-identification


## Data Preprocessing

In [5]:
# Clean raw files and save
hobbit_input_path = os.path.join(BASE_DIR, "data", "hobbit_raw.txt")
hobbit_output_path = os.path.join(BASE_DIR, "data", "hobbit_clean.txt")
hobbit = clean_text(hobbit_input_path, hobbit_output_path)

lw_input_path = os.path.join(BASE_DIR, "data", "lostworld_raw.txt")
low_output_path = os.path.join(BASE_DIR, "data", "lostworld_clean.txt")
lostworld = clean_text(lw_input_path, low_output_path)

## Train-Holdout Split

In [8]:
# Train-Holdout Split
hobbit_train, hobbit_holdout = split_train_holdout(hobbit)
lostworld_train, lostworld_holdout = split_train_holdout(lostworld)

# Save to files
hobbit_train_path = os.path.join(BASE_DIR, "texts", "hobbit_train.txt")
hobbit_holdout_path = os.path.join(BASE_DIR, "texts", "hobbit_holdout.txt")
lostworld_train_path = os.path.join(BASE_DIR, "texts", "lostworld_train.txt")
lostworld_holdout_path = os.path.join(BASE_DIR, "texts", "lostworld_holdout.txt")
paths = [hobbit_train_path, hobbit_holdout_path, lostworld_train_path, lostworld_holdout_path]

for path in paths:
    with open(path, "w", encoding="utf-8") as f:
        if "hobbit" in path:
            f.write(hobbit_train if "train" in path else hobbit_holdout)
        else:
            f.write(lostworld_train if "train" in path else lostworld_holdout)

## Tokenization

In [12]:
# First BPE Tokenizer Attempted using `Whitespace()` pre-tokenizer
class BPETokenizer:
    """Wraps HuggingFace's tokenizers library to provide a simple
    encode/decode/train interface for n-gram LM pipeline."""

    def __init__(self, vocab_size=5000, pre_tokenizer="whitespace"):
        self.vocab_size = vocab_size
        self.tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))

        if pre_tokenizer == "whitespace":
            self.tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
        elif pre_tokenizer == "byte_level":
            self.tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel()
        else:
            raise ValueError(f"Unknown pre_tokenizer: {pre_tokenizer}")

        self.special_tokens = ["<unk>", "<pad>", "<s>", "</s>"]

    def train(self, filepaths):
        """Train BPE on one or more text files (paths as a list)."""
        trainer = trainers.BpeTrainer(
            vocab_size=self.vocab_size,
            special_tokens=self.special_tokens,
        )
        self.tokenizer.train(filepaths, trainer)

    def encode(self, text):
        """Returns list of token ids."""
        return self.tokenizer.encode(text).ids

    def decode(self, ids):
        """Returns string from list of token ids."""
        return self.tokenizer.decode(ids)

    def get_vocab_size(self):
        return self.tokenizer.get_vocab_size()

    def save(self, path):
        self.tokenizer.save(path)

    def load(self, path):
        self.tokenizer = Tokenizer.from_file(path)

# Training
hobbit_train_path = os.path.join(BASE_DIR, "data", "hobbit_train.txt")
lostworld_train_path = os.path.join(BASE_DIR, "data", "lostworld_train.txt")
bpe = BPETokenizer(vocab_size=5000, pre_tokenizer="whitespace")
bpe.train([hobbit_train_path, lostworld_train_path])

# VALIDATION
print("Tokenizer Validation: The Hobbit")
sample = "Bilbo Baggins was a hobbit who lived in a hole in the ground."
ids = bpe.encode(sample)
print(ids)
print(bpe.decode(ids))
print("vocab size:", bpe.get_vocab_size(), "\n")

print("Tokenizer Validation: The Lost World")
sample = """
Well, at least
you are better than that herd of swine in Vienna, whose gregarious
grunt is, however, not more offensive than the isolated effort of the
British hog.
"""
ids = bpe.encode(sample)
print(ids)
print(bpe.decode(ids))
print("vocab size:", bpe.get_vocab_size())




Tokenizer Validation: The Hobbit
[239, 723, 117, 55, 474, 287, 1229, 88, 55, 1159, 88, 87, 685, 12]
Bilbo Baggins was a hobbit who lived in a hole in the ground .
vocab size: 5000 

Tokenizer Validation: The Lost World
[674, 10, 94, 981, 132, 201, 913, 357, 126, 1050, 58, 99, 1040, 425, 88, 4578, 3280, 10, 2156, 225, 1846, 741, 255, 2643, 105, 10, 1015, 10, 161, 275, 3923, 357, 87, 4110, 3161, 99, 87, 2606, 3552, 159, 61, 12]
Well , at least you are better than that her d of sw ine in Vien na , whose gre gar ious gr unt is , however , not more offensive than the isolated effort of the Br itish ho g .
vocab size: 5000


The decode is inserting a space between every token, even subword pieces that should be merged (e.g. "her d" instead of "herd"). This is the default behavior of the `Whitespace()` pre-tokenizer, so we move to the `ByteLevel` implementation.

After one iteration of `ByteLevel`, we discovered that any character that happened to not appear in training (e.g. "\n") will fall back to `<unk>` and cause incorrect spacing (merging words that should be separate). To fix, we specify that the trainer should start out with all 256 byte tokens for alphabet up front, instead of only what it happens to see in training corpus.

In [13]:
class BPETokenizer:
    def __init__(self, vocab_size=5000, pre_tokenizer="byte_level"):
        self.vocab_size = vocab_size
        self.tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))

        if pre_tokenizer == "byte_level":
            self.tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
            self.tokenizer.decoder = decoders.ByteLevel()
        elif pre_tokenizer == "whitespace":
            self.tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
            # no matching decoder so use default
        else:
            raise ValueError(f"Unknown pre_tokenizer: {pre_tokenizer}")

        self.special_tokens = ["<unk>", "<pad>", "<s>", "</s>"]

    def train(self, filepaths):
        """Train BPE on one or more text files (paths as a list)."""
        trainer = trainers.BpeTrainer(
            vocab_size=self.vocab_size,
            special_tokens=self.special_tokens,
            initial_alphabet=pre_tokenizers.ByteLevel.alphabet()
        )
        self.tokenizer.train(filepaths, trainer)

    def encode(self, text):
        """Normalize input and returns list of token ids."""
        text = re.sub(r"\s+", " ", text).strip()
        return self.tokenizer.encode(text).ids

    def decode(self, ids):
        """Returns string from list of token ids."""
        return self.tokenizer.decode(ids)

    def get_vocab_size(self):
        return self.tokenizer.get_vocab_size()

    def save(self, path):
        self.tokenizer.save(path)

    def load(self, path):
        self.tokenizer = Tokenizer.from_file(path)

# Training
hobbit_train_path = os.path.join(BASE_DIR, "data", "hobbit_train.txt")
lostworld_train_path = os.path.join(BASE_DIR, "data", "lostworld_train.txt")
bpe = BPETokenizer(vocab_size=5000, pre_tokenizer="byte_level")
bpe.train([hobbit_train_path, lostworld_train_path])

# VALIDATION
print("Tokenizer Validation: The Hobbit")
sample = "Bilbo Baggins was a hobbit who lived in a hole in the ground."
ids = bpe.encode(sample)
print(ids)
print(bpe.decode(ids))
print("vocab size:", bpe.get_vocab_size(), "\n")

print("Tokenizer Validation: The Lost World")
sample = """
Well, at least
you are better than that herd of swine in Vienna, whose gregarious
grunt is, however, not more offensive than the isolated effort of the
British hog.
"""
ids = bpe.encode(sample)
print(ids)
print(bpe.decode(ids))
print("vocab size:", bpe.get_vocab_size())




Tokenizer Validation: The Hobbit
[4654, 981, 309, 262, 735, 545, 1570, 296, 262, 1982, 296, 263, 1103, 17]
Bilbo Baggins was a hobbit who lived in a hole in the ground.
vocab size: 5000 

Tokenizer Validation: The Lost World
[1090, 15, 366, 1285, 332, 468, 1207, 604, 325, 1565, 71, 280, 662, 670, 296, 3364, 2146, 3726, 15, 2552, 753, 2707, 656, 537, 407, 87, 379, 15, 1311, 15, 360, 513, 616, 3905, 604, 263, 4650, 3304, 280, 263, 3217, 4342, 481, 74, 17]
Well, at least you are better than that herd of swine in Vienna, whose gregarious grunt is, however, not more offensive than the isolated effort of the British hog.
vocab size: 5000


## N-Gram Language Models

In [14]:
# Encode training files for N-Gram Language Models
hobbit_sentences = get_sentences(hobbit_train)
hobbit_encoded = encode_sentences(hobbit_sentences, bpe)
lostworld_sentences = get_sentences(lostworld_train)
lostworld_encoded = encode_sentences(lostworld_sentences, bpe)

In [24]:
# N-Gram model
class NGramModel:
    "Unigram, bigram and trigram counts with add-k smoothing"

    def __init__(self, vocab_size):
        self.vocab_size = vocab_size
        self.unigram_counts = Counter()
        self.bigram_counts = Counter()
        self.trigram_counts = Counter()
        self.bigram_context_counts = Counter()
        self.trigram_context_counts = Counter()

    def train(self, encoded_sentences):
        """encoded_sentences: list of token id lists wrapped with <s> and </s>"""
        for sent in encoded_sentences:
            for i, token in enumerate(sent):
                self.unigram_counts[token] += 1

                if i >= 1:
                    bigram = (sent[i-1], sent[i])
                    self.bigram_counts[bigram] += 1
                    self.bigram_context_counts[(sent[i-1]),] += 1

                if i >= 2:
                    trigram = (sent[i-2], sent[i-1], sent[i])
                    self.trigram_counts[trigram] += 1
                    self.trigram_context_counts[(sent[i-2], sent[i-1])] += 1

    def bigram_prob(self, w1, w2, k=1.0):
        """P(w2 | w1) with add-k smoothing"""
        count_bigram = self.bigram_counts[(w1, w2)]
        count_context = self.bigram_context_counts[(w1,)]
        return (count_bigram + k) / (count_context + k * self.vocab_size)

    def trigram_prob(self, w1, w2, w3, k=1.0):
        """P(w3 | w1, w2) with add-k smoothing"""
        count_trigram = self.trigram_counts[(w1, w2, w3)]
        count_context = self.trigram_context_counts[(w1,w2)]
        return (count_trigram + k) / (count_context + k * self.vocab_size)

    def neg_log_prob(self, sent, n=2, k=1.0):
        """Returns total -logP(sentence) using n_gram model (n=2 for bigram, n=3 for trigram)"""
        total = 0.0
        for i in range(1, len(sent)) if n==2 else range(2, len(sent)):
            if n==2:
                p = self.bigram_prob(sent[i-1], sent[i], k)
            elif n==3:
                p = self.trigram_prob(sent[i-2], sent[i-1], sent[i], k)
            else:
                raise ValueError("n must be 2 (bigram) or 3 (trigram)")
            total += -math.log(p)
        return total

    def perplexity(self, sentences, n=2, k=1.0):
        """Computes perplexity over list of encoded sentences (list of token id lists)"""
        total_neg_log_prob = 0.0
        total_tokens = 0

        for sent in sentences:
            total_neg_log_prob += self.neg_log_prob(sent, n=n, k=k)
            # count predicted tokens, excl. <s>
            # same logic inside neg_log_prob: start at 1 for bigram, 2 for trigram
            n_predicted = len(sent)-1 if n==2 else len(sent)-2
            total_tokens += n_predicted

        avg_neg_log_prob = total_neg_log_prob / total_tokens
        return math.exp(avg_neg_log_prob)

# Validation
# Test on Tolkien
print("NGramModel Validation: Tolkien")
ngram = NGramModel(vocab_size=bpe.get_vocab_size())
ngram.train(hobbit_encoded)
test_seq = random.sample(hobbit_encoded,1)[0]
print(test_seq)
print("bigram neg log prob:", ngram.neg_log_prob(test_seq, n=2, k=1.0))
print("trigram neg log prob:", ngram.neg_log_prob(test_seq, n=3, k=1.0))
print(bpe.decode(test_seq), "\n")

# Test on Doyle
print("NGramModel Validation: Conan Doyle")
ngram = NGramModel(vocab_size=bpe.get_vocab_size())
ngram.train(lostworld_encoded)
test_seq = random.sample(lostworld_encoded,1)[0]
print(test_seq)
print("bigram neg log prob:", ngram.neg_log_prob(test_seq, n=2, k=1.0))
print("trigram neg log prob:", ngram.neg_log_prob(test_seq, n=3, k=1.0))
print(bpe.decode(test_seq))

NGramModel Validation: Tolkien
[2, 1706, 619, 360, 530, 317, 280, 997, 15, 400, 317, 309, 360, 418, 1639, 571, 300, 1188, 15, 513, 432, 1336, 15, 532, 309, 296, 1153, 280, 396, 17, 3]
bigram neg log prob: 170.38756375818
trigram neg log prob: 198.22586159395934
He did not like it of course, but it was not so bad now he knew, more or less, what was in front of him. 

NGramModel Validation: Conan Doyle
[2, 44, 934, 388, 1007, 290, 15, 400, 354, 1659, 2436, 86, 275, 1567, 296, 410, 1157, 275, 478, 1957, 585, 1465, 287, 1024, 609, 15, 275, 2562, 368, 463, 2420, 264, 10, 426, 1157, 348, 3015, 516, 15, 418, 354, 468, 2352, 280, 535, 1566, 325, 301, 382, 905, 296, 393, 4988, 264, 583, 17, 3]
bigram neg log prob: 371.9372001329504
trigram neg log prob: 421.545989335452
I call them apes, but they carried sticks and stones in their hands and jabbered talk to each other, and ended up by tyin' our hands with creepers, so they are ahead of any beast that I have seen in my wanderin's.


*Note:* Abbreviation-splitting limitation. Finding "." + space + Capital Letter means that titles such as "Mr. Smith" are split into two sentences.

In [25]:
# Perplexity
## Tolkien
ngram = NGramModel(vocab_size=bpe.get_vocab_size())
ngram.train(hobbit_encoded)
hobbit_holdout_sentences = get_sentences(hobbit_holdout)
hobbit_holdout_encoded = encode_sentences(hobbit_holdout_sentences, bpe)
bigram_ppl = ngram.perplexity(hobbit_holdout_encoded, n=2, k=1.0)
trigram_ppl = ngram.perplexity(hobbit_holdout_encoded, n=3, k=1.0)
print("Hobbit")
print("bigram perplexity:", bigram_ppl)
print("trigram perplexity:", trigram_ppl)

## Doyle
ngram = NGramModel(vocab_size=bpe.get_vocab_size())
ngram.train(lostworld_encoded)
lostworld_holdout_sentences = get_sentences(lostworld_holdout)
lostworld_holdout_encoded = encode_sentences(lostworld_holdout_sentences, bpe)
bigram_ppl = ngram.perplexity(lostworld_holdout_encoded, n=2, k=1.0)
trigram_ppl = ngram.perplexity(lostworld_holdout_encoded, n=3, k=1.0)
print("\nLost World")
print("bigram perplexity:", bigram_ppl)
print("trigram perplexity:", trigram_ppl)

Hobbit
bigram perplexity: 915.2484905503446
trigram perplexity: 3244.3203950290495

Lost World
bigram perplexity: 1171.0410086794275
trigram perplexity: 3574.9187071801944


**Observations**
* Trigram performs much worse than bigram with k=1.0
* Number of distinct trigram contexts is huge compared to how many times each actually appears, so smoothing dominates probability estimate

## Hyperparameter Tuning

### K Value for Smoothing

In [26]:
# K-sweep: testing different k values
k_vals = [1.0, 0.5, 0.1, 0.01, 0.001, 0.0001, 0.00001, 0.000001]

# Tolkien
ngram = NGramModel(vocab_size=bpe.get_vocab_size())
ngram.train(hobbit_encoded)
print("Hobbit")
for k in k_vals:
    bigram_ppl = ngram.perplexity(hobbit_holdout_encoded, n=2, k=k)
    trigram_ppl = ngram.perplexity(hobbit_holdout_encoded, n=3, k=k)
    print(f"k={k:<6} bigram={bigram_ppl:9.2f}   trigram={trigram_ppl:9.2f}")

# Doyle
ngram = NGramModel(vocab_size=bpe.get_vocab_size())
ngram.train(lostworld_encoded)
print("\nLost World")
for k in k_vals:
    bigram_ppl = ngram.perplexity(lostworld_holdout_encoded, n=2, k=k)
    trigram_ppl = ngram.perplexity(lostworld_holdout_encoded, n=3, k=k)
    print(f"k={k:<6} bigram={bigram_ppl:9.2f}   trigram={trigram_ppl:9.2f}")

Hobbit
k=1.0    bigram=   915.25   trigram=  3244.32
k=0.5    bigram=   684.38   trigram=  2809.16
k=0.1    bigram=   387.96   trigram=  1947.10
k=0.01   bigram=   273.99   trigram=  1283.29
k=0.001  bigram=   346.57   trigram=  1237.18
k=0.0001 bigram=   628.70   trigram=  2010.28
k=1e-05  bigram=  1266.63   trigram=  4500.52
k=1e-06  bigram=  2591.69   trigram= 10824.14

Lost World
k=1.0    bigram=  1171.04   trigram=  3574.92
k=0.5    bigram=   879.95   trigram=  3168.45
k=0.1    bigram=   488.79   trigram=  2306.41
k=0.01   bigram=   326.04   trigram=  1582.17
k=0.001  bigram=   409.20   trigram=  1518.34
k=0.0001 bigram=   779.24   trigram=  2457.48
k=1e-05  bigram=  1676.60   trigram=  5623.86
k=1e-06  bigram=  3672.38   trigram= 13926.26


**Observations**
* The bigram models outperforms the trigram model at every value of k

*Bigrams*: perplexity drops from k=1.0 to k=0.01, then increases at k=0.001 for both books
* k = 0.01 is ideal hyperparameter for bigrams

*Trigrams*: perplexity drops from k=1.0 to k=0.001, then increases for both books
* k = 0.001 is ideal hyperparameter for trigrams

|Model|Best k|Best perplexity (Hobbit)|Best perplexity (Lost World)|
|---|---|---|---|
|Bigram|0.01|273.99|326.04|
|Trigram|0.001|1237.18|1518.34|

### Vocabulary Size

In [27]:
# Testing different vocabulary sizes
vocab_sizes = [100, 250, 500, 1000, 2000, 5000, 10000, 20000]
results = []

for vsize in vocab_sizes:
    # Retrain tokenizer at this vocab size
    bpe_v = BPETokenizer(vocab_size=vsize, pre_tokenizer="byte_level")
    bpe_v.train([hobbit_train_path, lostworld_train_path])

    # Re-tokenize everything with this tokenizer
    hobbit_seq = encode_sentences(get_sentences(hobbit_train), bpe_v)
    hobbit_hold_seq = encode_sentences(get_sentences(hobbit_holdout), bpe_v)
    lw_seq = encode_sentences(get_sentences(lostworld_train), bpe_v)
    lw_hold_seq = encode_sentences(get_sentences(lostworld_holdout), bpe_v)

    # Train bigram models (using your chosen best config: bigram, k=0.01)
    tolkien_m = NGramModel(vocab_size=bpe_v.get_vocab_size())
    tolkien_m.train(hobbit_seq)

    doyle_m = NGramModel(vocab_size=bpe_v.get_vocab_size())
    doyle_m.train(lw_seq)

    # Evaluate perplexity on each holdout
    hobbit_ppl = tolkien_m.perplexity(hobbit_hold_seq, n=2, k=0.01)
    lw_ppl = doyle_m.perplexity(lw_hold_seq, n=2, k=0.01)

    results.append((vsize, hobbit_ppl, lw_ppl))
    print(f"vocab_size={vsize:<6} Hobbit ppl={hobbit_ppl:9.2f}   Lost World ppl={lw_ppl:9.2f}")




vocab_size=100    Hobbit ppl=    10.07   Lost World ppl=    11.44



vocab_size=250    Hobbit ppl=    10.07   Lost World ppl=    11.44



vocab_size=500    Hobbit ppl=    35.51   Lost World ppl=    40.70



vocab_size=1000   Hobbit ppl=    69.12   Lost World ppl=    74.67



vocab_size=2000   Hobbit ppl=   122.10   Lost World ppl=   131.83



vocab_size=5000   Hobbit ppl=   273.99   Lost World ppl=   326.04



vocab_size=10000  Hobbit ppl=   456.38   Lost World ppl=   646.27



vocab_size=20000  Hobbit ppl=   642.10   Lost World ppl=  1085.83


**Observations**
* vocab_size=100 and 250 perform identically because the ByteLevel tokenizer setup initializes with all 256 possible byte values before any BPE merges are learned, as well as 4 special tokens, so that's 260 vocab entries to start with
* vocab_size=100 or 250 are not valid configurations
* Raw perplexity comparisons across vocab sizes is not a valid evaluation metric for ideal vocabulary size due to differences in token granularity

Keeping vocab_size = 5000 as reasonable middle ground.

## Author Classification

In [29]:
# Final Tokenizer
bpe = BPETokenizer(vocab_size=5000, pre_tokenizer="byte_level")
bpe.train([hobbit_train_path, lostworld_train_path])

# Re-tokenize training and holdout sets
hobbit_encoded = encode_sentences(get_sentences(hobbit_train), bpe)
hobbit_holdout_encoded = encode_sentences(get_sentences(hobbit_holdout), bpe)
lostworld_encoded = encode_sentences(get_sentences(lostworld_train), bpe)
lostworld_holdout_encoded = encode_sentences(get_sentences(lostworld_holdout), bpe)

# Train final models for each author
tolkien_model = NGramModel(vocab_size=bpe.get_vocab_size())
tolkien_model.train(hobbit_encoded)
doyle_model = NGramModel(vocab_size=bpe.get_vocab_size())
doyle_model.train(lostworld_encoded)

In [30]:
# Test on holdout sets
pred, t_ppl, d_ppl = predict_author(hobbit_holdout, tolkien_model, doyle_model, bpe)
print(f"Hobbit holdout -> Prediction: {pred}  (Tolkien ppl={t_ppl:.2f}, Doyle ppl={d_ppl:.2f})")

pred, t_ppl, d_ppl = predict_author(lostworld_holdout, tolkien_model, doyle_model, bpe)
print(f"Lost World holdout -> Prediction: {pred}  (Tolkien ppl={t_ppl:.2f}, Doyle ppl={d_ppl:.2f})")

Hobbit holdout -> Prediction: J.R.R. Tolkien  (Tolkien ppl=273.99, Doyle ppl=821.26)
Lost World holdout -> Prediction: Arthur Conan Doyle  (Tolkien ppl=852.19, Doyle ppl=326.04)


In [31]:
# Test on different passage sizes (in sentences)
def chunk_sentences(sentences, chunk_size=5):
    """Groups sentences into short passages of chunk_size sentences each."""
    return [" ".join(sentences[i:i+chunk_size]) for i in range(0, len(sentences), chunk_size)]

hobbit_holdout_sentences = get_sentences(hobbit_holdout)
lostworld_holdout_sentences = get_sentences(lostworld_holdout)
hobbit_chunks = chunk_sentences(hobbit_holdout_sentences, chunk_size=5)
lostworld_chunks = chunk_sentences(lostworld_holdout_sentences, chunk_size=5)

correct = 0
total = 0

for chunk in hobbit_chunks:
    pred, _, _ = predict_author(chunk, tolkien_model, doyle_model, bpe)
    correct += (pred == "J.R.R. Tolkien")
    total += 1

for chunk in lostworld_chunks:
    pred, _, _ = predict_author(chunk, tolkien_model, doyle_model, bpe)
    correct += (pred == "Arthur Conan Doyle")
    total += 1

print(f"Accuracy on holdout chunks: {correct}/{total} = {correct/total:.2%}")

Accuracy on holdout chunks: 142/142 = 100.00%


In [32]:
# Test robustness of pipeline against different chunk sizes
for chunk_size in [1, 2, 3, 5, 10]:
    hobbit_chunks = chunk_sentences(hobbit_holdout_sentences, chunk_size=chunk_size)
    lostworld_chunks = chunk_sentences(lostworld_holdout_sentences, chunk_size=chunk_size)

    correct = 0
    total = 0
    for chunk in hobbit_chunks:
        pred, _, _ = predict_author(chunk, tolkien_model, doyle_model, bpe)
        correct += (pred == "J.R.R. Tolkien")
        total += 1
    for chunk in lostworld_chunks:
        pred, _, _ = predict_author(chunk, tolkien_model, doyle_model, bpe)
        correct += (pred == "Arthur Conan Doyle")
        total += 1

    print(f"chunk_size={chunk_size:<3} accuracy={correct}/{total} = {correct/total:.2%}")

chunk_size=1   accuracy=638/705 = 90.50%
chunk_size=2   accuracy=346/353 = 98.02%
chunk_size=3   accuracy=235/235 = 100.00%
chunk_size=5   accuracy=142/142 = 100.00%
chunk_size=10  accuracy=71/71 = 100.00%


**Observations**
* Classification accuracy reaches 100% on holdout passages of 3 or more sentences, with diminished but still strong performance (90.5%) on single-sentence passages.
* This suggests the model needs a minimum amount of context to reliably capture the writer's style, but that minimum is small.

In [ ]:
# Inspect Hobbit Misclassifications (1-sentence)
misclassified = []
for chunk in chunk_sentences(hobbit_holdout_sentences, chunk_size=1):
    pred, t_ppl, d_ppl = predict_author(chunk, tolkien_model, doyle_model, bpe)
    if pred != "Tolkien":
        misclassified.append((chunk, t_ppl, d_ppl))

print(f"{len(misclassified)} misclassified Hobbit sentences:")
for sent, t, d in misclassified[:20]:
    print(f"  '{sent}'  (Tolkien ppl={t:.1f}, Doyle ppl={d:.1f})")

396 misclassified Hobbit sentences:
  'You undersized — burglar!” he shouted at a loss for words, and he shook poor Bilbo like a rabbit. “By the beard of Durin!'  (Tolkien ppl=425.6, Doyle ppl=1369.4)
  'I wish I had Gandalf here!'  (Tolkien ppl=151.9, Doyle ppl=205.8)
  'Curse him for his choice of you!'  (Tolkien ppl=113.7, Doyle ppl=231.1)
  'May his beard wither!'  (Tolkien ppl=160.4, Doyle ppl=368.1)
  'As for you I will throw you to the rocks!” he cried and lifted Bilbo in his arms. “Stay!'  (Tolkien ppl=171.4, Doyle ppl=706.0)
  'Your wish is granted!” said a voice.'  (Tolkien ppl=261.7, Doyle ppl=2070.8)
  'The old man with the casket threw aside his hood and cloak. “Here is Gandalf!'  (Tolkien ppl=163.0, Doyle ppl=1221.6)
  'And none too soon it seems.'  (Tolkien ppl=165.1, Doyle ppl=1022.3)
  'If you don’t like my Burglar, please don’t damage him.'  (Tolkien ppl=76.6, Doyle ppl=1969.5)
  'Put him down, and listen first to what he has to say!” “You all seem in league!” said Th

In [ ]:
# Inspect Lost World Misclassifications (1-sentence)
misclassified = []
for chunk in chunk_sentences(lostworld_holdout_sentences, chunk_size=1):
    pred, t_ppl, d_ppl = predict_author(chunk, tolkien_model, doyle_model, bpe)
    if pred != "Tolkien":
        misclassified.append((chunk, t_ppl, d_ppl))

print(f"{len(misclassified)} misclassified Lost World sentences:")
for sent, t, d in misclassified[:20]:
    print(f"  '{sent}'  (Tolkien ppl={t:.1f}, Doyle ppl={d:.1f})")

309 misclassified Lost World sentences:
  'Then suddenly he stretched out his hand and seized the puzzle. "By George!" he cried, "I believe I've got it.'  (Tolkien ppl=354.1, Doyle ppl=121.3)
  'The boy guessed right the very first time.'  (Tolkien ppl=304.5, Doyle ppl=447.3)
  'See here!'  (Tolkien ppl=230.6, Doyle ppl=213.3)
  'How many marks are on that paper?'  (Tolkien ppl=2242.5, Doyle ppl=573.0)
  'Eighteen.'  (Tolkien ppl=399.4, Doyle ppl=399.4)
  'Well, if you come to think of it there are eighteen cave openings on the hill-side above us." "He pointed up to the caves when he gave it to me," said I. "Well, that settles it.'  (Tolkien ppl=386.6, Doyle ppl=149.5)
  'This is a chart of the caves.'  (Tolkien ppl=93.7, Doyle ppl=30.6)
  'What!'  (Tolkien ppl=26.9, Doyle ppl=114.8)
  'Eighteen of them all in a row, some short, some deep, some branching, same as we saw them.'  (Tolkien ppl=661.1, Doyle ppl=236.5)
  'It's a map, and here's a cross on it.'  (Tolkien ppl=163.7, Doyle ppl

In [35]:
# Calculate dialogue fraction for each novel
def dialogue_fraction(sentences):
    dialogue = sum(1 for s in sentences if any(q in s for q in ['"', '“', '”']))
    return dialogue / len(sentences)

print("Hobbit dialogue fraction:", dialogue_fraction(get_sentences(hobbit_train)))
print("Lost World dialogue fraction:", dialogue_fraction(get_sentences(lostworld_train)))

Hobbit dialogue fraction: 0.21692825112107622
Lost World dialogue fraction: 0.2005740940078938


**Observations**
* Misclassified Tolkien sentences are predominantly dialogue, but misclassified Doyle lines are a mix of dialogue and prose
* Hypothesize that difference is because Tolkien writes in third person, but Doyle writes in first person, so there's less of a distinction between Doyle's dialogue and prose

In [36]:
# Calculate use of first person
def first_person_rate(sentences):
    """Fraction of non-dialogue sentences containing a first-person pronoun."""
    pronouns = r"\b(I|me|my|mine|myself)\b"
    narration = [s for s in sentences if not any(q in s for q in ['"', '“', '”'])]
    if not narration:
        return 0
    hits = sum(1 for s in narration if re.search(pronouns, s))
    return hits / len(narration)

print("Hobbit narration first-person rate:", first_person_rate(get_sentences(hobbit_train)))
print("Lost World narration first-person rate:", first_person_rate(get_sentences(lostworld_train)))

Hobbit narration first-person rate: 0.09055118110236221
Lost World narration first-person rate: 0.35637342908438063
